# Phase 11: Targeted Improvements — Results

**This notebook loads pre-computed JSON result files only. No training is performed.**

Three improvements were implemented and evaluated:
- **11A**: Confidence filtering on V2 and V2.1 fusion models
- **11B**: Expanded macro encoder (32d → 128d), producing V2.1
- **11C**: Weekly long-short backtest with transaction costs

Result files:
- `models/phase11_v2_1_results.json` — V2.1 fusion test metrics
- `models/phase11_confidence_results.json` — V2 confidence filtering sweep
- `models/phase11_confidence_v2_1_results.json` — V2.1 confidence filtering sweep
- `models/phase11_backtest_results.json` — Weekly rebalancing backtest

In [ ]:
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

def load_json(path):
    with open(path) as f:
        return json.load(f)

res_v2_1      = load_json('../models/phase11_v2_1_results.json')
res_conf_v2   = load_json('../models/phase11_confidence_results.json')
res_conf_v2_1 = load_json('../models/phase11_confidence_v2_1_results.json')
res_backtest  = load_json('../models/phase11_backtest_results.json')

# Convenience aliases
fusion_v2_1 = res_v2_1['fusion_v2_1']

print('All result files loaded successfully.')
print(f"  V2.1 fusion: AUC={fusion_v2_1['direction_auc']:.4f},  macro gate={fusion_v2_1['gate_weights']['macro']:.1%}")
print(f"  V2 best conf AUC: {res_conf_v2['best_filtered_auc']:.4f} @ thresh={res_conf_v2['best_filter_threshold']}")
print(f"  V2.1 best conf AUC: {res_conf_v2_1['best_filtered_auc']:.4f} @ thresh={res_conf_v2_1['best_filter_threshold']}")
print(f"  Backtest equal_weight 0bp: return={res_backtest['equal_weight']['0']['total_return']*100:.1f}%")

## 1. Phase 11B: V2.1 Macro Encoder Expansion (32d → 128d)

In [ ]:
# V2 baseline from Phase 10
v2_baseline = {
    'auc': 0.5989, 'acc': 0.5920, 'vol_r2': 0.3350,
    'gate_price': 0.236, 'gate_gat': 0.412,
    'gate_doc': 0.227,   'gate_macro': 0.126,
}

gw = fusion_v2_1['gate_weights']

rows = [
    ['Direction AUC', f"{v2_baseline['auc']:.4f}",    f"{fusion_v2_1['direction_auc']:.4f}",  f"{fusion_v2_1['direction_auc']-v2_baseline['auc']:+.4f}"],
    ['Direction Acc', f"{v2_baseline['acc']:.4f}",    f"{fusion_v2_1['direction_acc']:.4f}",  f"{fusion_v2_1['direction_acc']-v2_baseline['acc']:+.4f}"],
    ['Direction F1',  'N/A',                           f"{fusion_v2_1['direction_f1']:.4f}",   'N/A'],
    ['Vol R²',        f"{v2_baseline['vol_r2']:.4f}", f"{fusion_v2_1['vol_r2']:.4f}",          f"{fusion_v2_1['vol_r2']-v2_baseline['vol_r2']:+.4f}"],
    ['Gate: Price',   f"{v2_baseline['gate_price']:.1%}", f"{gw['price']:.1%}", f"{gw['price']-v2_baseline['gate_price']:+.1%}"],
    ['Gate: GAT',     f"{v2_baseline['gate_gat']:.1%}",   f"{gw['gat']:.1%}",   f"{gw['gat']-v2_baseline['gate_gat']:+.1%}"],
    ['Gate: Doc',     f"{v2_baseline['gate_doc']:.1%}",   f"{gw['doc']:.1%}",   f"{gw['doc']-v2_baseline['gate_doc']:+.1%}"],
    ['Gate: Macro',   f"{v2_baseline['gate_macro']:.1%}", f"{gw['macro']:.1%}", f"{gw['macro']-v2_baseline['gate_macro']:+.1%}"],
    ['Params',        '~550K', f"{fusion_v2_1['params']:,}", 'N/A'],
]

df_cmp = pd.DataFrame(rows, columns=['Metric', 'V2 (macro 32d)', 'V2.1 (macro 128d)', 'Delta'])
print('=== V2 vs V2.1 Comparison ===')
print(df_cmp.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

modalities = ['Price', 'GAT', 'Doc', 'Macro']
v2_gates   = [v2_baseline['gate_price'], v2_baseline['gate_gat'],
              v2_baseline['gate_doc'],   v2_baseline['gate_macro']]
v2_1_gates = [gw['price'], gw['gat'], gw['doc'], gw['macro']]

x = np.arange(len(modalities))
width = 0.35

ax = axes[0]
b1 = ax.bar(x - width/2, v2_gates,   width, label='V2 (32d)',   color='steelblue', alpha=0.85)
b2 = ax.bar(x + width/2, v2_1_gates, width, label='V2.1 (128d)', color='darkorange', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(modalities)
ax.set_ylabel('Gate Weight')
ax.set_title('Modality Gate Weights: V2 vs V2.1')
ax.legend()
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
for bar in list(b1) + list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
            f'{bar.get_height():.1%}', ha='center', va='bottom', fontsize=9)

ax2 = axes[1]
metrics   = ['AUC', 'Accuracy', 'Vol R²']
v2_vals   = [v2_baseline['auc'], v2_baseline['acc'], v2_baseline['vol_r2']]
v2_1_vals = [fusion_v2_1['direction_auc'], fusion_v2_1['direction_acc'], fusion_v2_1['vol_r2']]
x2 = np.arange(len(metrics))
ax2.bar(x2 - width/2, v2_vals,   width, label='V2 (32d)',   color='steelblue', alpha=0.85)
ax2.bar(x2 + width/2, v2_1_vals, width, label='V2.1 (128d)', color='darkorange', alpha=0.85)
ax2.set_xticks(x2); ax2.set_xticklabels(metrics)
ax2.set_ylabel('Score')
ax2.set_title('Model Performance: V2 vs V2.1')
ax2.legend()
ax2.set_ylim(0, 0.75)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../models/phase11_v2_comparison.png', bbox_inches='tight')
plt.show()
print('Figure saved to models/phase11_v2_comparison.png')

## 2. Phase 11A: Confidence Filtering

Threshold sweep: keep sample only if `max(P(UP), P(DOWN)) ≥ threshold`.

In [ ]:
def sweep_to_df(res, label):
    rows = []
    for entry in res['sweep']:
        rows.append({
            'Model':     label,
            'Threshold': entry['threshold'],
            'N':         entry['N'],
            'Coverage':  entry['coverage_pct'] / 100.0,   # fraction
            'AUC':       entry.get('auc', float('nan')),
            'Acc':       entry.get('accuracy', float('nan')),
            'F1':        entry.get('f1', float('nan')),
        })
    return pd.DataFrame(rows)

df_cv2   = sweep_to_df(res_conf_v2,   'V2')
df_cv2_1 = sweep_to_df(res_conf_v2_1, 'V2.1')

print('=== V2 Confidence Filtering Sweep ===')
print(df_cv2[['Threshold','N','Coverage','AUC','Acc','F1']].to_string(index=False, float_format='{:.4f}'.format))
print()
print('=== V2.1 Confidence Filtering Sweep ===')
print(df_cv2_1[['Threshold','N','Coverage','AUC','Acc','F1']].to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, df, res_d, label, color in [
    (axes[0], df_cv2,   res_conf_v2,   'V2',   'steelblue'),
    (axes[1], df_cv2_1, res_conf_v2_1, 'V2.1', 'darkorange'),
]:
    valid = df.dropna(subset=['AUC'])
    ax.plot(valid['Coverage']*100, valid['AUC'], 'o-', color=color,
            linewidth=2, markersize=7, label='AUC')
    ax.axhline(0.50, color='gray', linestyle='--', linewidth=1, label='AUC=0.50 (random)')
    best_idx = valid['AUC'].idxmax()
    best_row = valid.loc[best_idx]
    ax.scatter([best_row['Coverage']*100], [best_row['AUC']], color='red', zorder=5, s=120,
               label=f"Best={best_row['AUC']:.4f} @ {best_row['Coverage']*100:.0f}%")
    ax.set_xlabel('Coverage (%)')
    ax.set_ylabel('AUC')
    ax.set_title(f'{label}: AUC vs Coverage')
    ax.legend(fontsize=9)
    ax.set_xlim(0, 110)
    ax.set_ylim(0.45, 0.70)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../models/phase11_confidence_sweep.png', bbox_inches='tight')
plt.show()
print('Figure saved to models/phase11_confidence_sweep.png')

In [ ]:
print('=== Confidence Filtering Summary ===')
for res_d, df, label in [(res_conf_v2, df_cv2, 'V2  '), (res_conf_v2_1, df_cv2_1, 'V2.1')]:
    full_auc = res_d['full_auc']
    best_auc = res_d['best_filtered_auc']
    best_cov = res_d['best_filtered_coverage']
    best_thr = res_d['best_filter_threshold']
    improved = best_auc > full_auc
    print(f"{label} full coverage (thresh=0.50): AUC = {full_auc:.4f}")
    print(f"{label} best filtered (thresh={best_thr:.2f}, {best_cov*100:.0f}% cov): AUC = {best_auc:.4f}",
          f"({'improved +' if improved else 'no improvement, '}delta={best_auc-full_auc:+.4f})")
    print()

## 3. Phase 11C: Weekly Long-Short Backtest

In [ ]:
bt_rows = []
for sizing in ['equal_weight', 'confidence_weighted']:
    for cost_str, m in res_backtest[sizing].items():
        bt_rows.append({
            'Sizing':      'Equal' if sizing == 'equal_weight' else 'Conf-Wt',
            'Cost (bps)':  int(cost_str),
            'Total Ret':   f"{m['total_return']*100:.1f}%",
            'Annual Ret':  f"{m['annual_return']*100:.1f}%",
            'Sharpe':      f"{m['sharpe']:.3f}",
            'Max DD':      f"{m['max_drawdown']*100:.1f}%",
            'Win Rate':    f"{m['win_rate']*100:.1f}%",
            'Turnover':    f"{m['avg_turnover']*100:.1f}%",
        })

df_bt = pd.DataFrame(bt_rows)
print('=== Weekly Long-Short Backtest Results (V2 Model, Test Set) ===')
print('Universe: 10 tickers   |   Top-2 Long / Bottom-2 Short')
print('Rebalance: every 5 trading days (weekly)   |   384 trading days, 96 rebalances')
print()
print(df_bt.to_string(index=False))
print()
print(f"Break-even cost (equal weight):      {res_backtest['equal_weight_break_even_bps']} bps")
print(f"Break-even cost (conf-weighted):     {res_backtest['confidence_weighted_break_even_bps']} bps")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
costs = [0, 5, 10, 20]

ax = axes[0]
ew_sharpes = [res_backtest['equal_weight'][str(c)]['sharpe'] for c in costs]
cw_sharpes = [res_backtest['confidence_weighted'][str(c)]['sharpe'] for c in costs]
ax.plot(costs, ew_sharpes, 'o-', color='steelblue',  linewidth=2, markersize=8, label='Equal Weight')
ax.plot(costs, cw_sharpes, 's--', color='darkorange', linewidth=2, markersize=8, label='Conf-Weighted')
ax.axhline(0, color='black', linewidth=0.8, linestyle=':')
ax.set_xlabel('Transaction Cost (bps)')
ax.set_ylabel('Sharpe Ratio')
ax.set_title('Sharpe Ratio vs Transaction Cost')
ax.legend(); ax.grid(alpha=0.3); ax.set_xticks(costs)

ax2 = axes[1]
ew_returns = [res_backtest['equal_weight'][str(c)]['total_return']*100 for c in costs]
cw_returns = [res_backtest['confidence_weighted'][str(c)]['total_return']*100 for c in costs]
ax2.bar(np.array(costs)-1, ew_returns, width=2, color='steelblue',  alpha=0.85, label='Equal Weight')
ax2.bar(np.array(costs)+1, cw_returns, width=2, color='darkorange', alpha=0.85, label='Conf-Weighted')
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_xlabel('Transaction Cost (bps)')
ax2.set_ylabel('Total Return (%)')
ax2.set_title('Total Return vs Transaction Cost')
ax2.legend(); ax2.grid(alpha=0.3, axis='y'); ax2.set_xticks(costs)

plt.tight_layout()
plt.savefig('../models/phase11_backtest_chart.png', bbox_inches='tight')
plt.show()
print('Figure saved to models/phase11_backtest_chart.png')

## 4. Phase 11 Master Summary

In [ ]:
print('=' * 65)
print('PHASE 11 MASTER SUMMARY')
print('=' * 65)

# AUC summary
v2_full_auc    = 0.5989
v2_1_full_auc  = fusion_v2_1['direction_auc']
v2_best_f_auc  = res_conf_v2['best_filtered_auc']
v2_1_best_f_auc = res_conf_v2_1['best_filtered_auc']
v2_1_best_thr  = res_conf_v2_1['best_filter_threshold']
v2_1_best_cov  = res_conf_v2_1['best_filtered_coverage']

print()
print('--- AUC Results ---')
print(f'  V2   full (macro 32d):            AUC = {v2_full_auc:.4f}  [baseline]')
print(f'  V2.1 full (macro 128d):           AUC = {v2_1_full_auc:.4f}  (Δ={v2_1_full_auc-v2_full_auc:+.4f})')
print(f'  V2   + confidence filter:         AUC = {v2_best_f_auc:.4f}  (no improvement over full)')
print(f'  V2.1 + confidence filter:         AUC = {v2_1_best_f_auc:.4f}  ({v2_1_best_cov*100:.0f}% coverage, thresh={v2_1_best_thr})')

print()
print('--- Macro Expansion Impact ---')
print(f'  Macro gate weight: {v2_baseline["gate_macro"]:.1%} → {gw["macro"]:.1%}  (x{gw["macro"]/v2_baseline["gate_macro"]:.1f})')
print(f'  Vol R²:            {v2_baseline["vol_r2"]:.4f}  → {fusion_v2_1["vol_r2"]:.4f}  (WORSE by {fusion_v2_1["vol_r2"]-v2_baseline["vol_r2"]:+.4f})')
print(f'  Verdict: Macro expansion improved gate engagement but hurt vol regression.')
print(f'           Directional AUC marginally worse. V2 remains best overall.')

print()
print('--- Confidence Filtering Summary ---')
print(f'  V2:   Filtering degrades AUC. Model has poor calibration.')
print(f'  V2.1: Filtering improves AUC by {v2_1_best_f_auc-v2_1_full_auc:+.4f} at {v2_1_best_cov*100:.0f}% coverage.')
print(f'  V2.1 is better calibrated — wider confidence spread.')

print()
bt_ew_0  = res_backtest['equal_weight']['0']
bt_ew_10 = res_backtest['equal_weight']['10']
print('--- Weekly Backtest (V2 Model, Test Set) ---')
print(f'  Period: {bt_ew_0["n_trading_days"]} trading days, {bt_ew_0["n_rebalances"]} rebalances')
print(f'  0 bp:   return={bt_ew_0["total_return"]*100:.1f}%, sharpe={bt_ew_0["sharpe"]:.3f}, win={bt_ew_0["win_rate"]*100:.1f}%')
print(f'  10 bp:  return={bt_ew_10["total_return"]*100:.1f}%, sharpe={bt_ew_10["sharpe"]:.3f}, win={bt_ew_10["win_rate"]*100:.1f}%')
print(f'  Break-even cost: {res_backtest["equal_weight_break_even_bps"]} bps')
print(f'  Avg weekly turnover: {bt_ew_0["avg_turnover"]*100:.1f}%')

print()
print('--- GPU Optimization (Step 1) ---')
print('  cuDNN benchmark: ON   TF32 matmul: ON')
print('  DataLoader: num_workers=0 on Windows (spawn overhead avoidance)')
print('  FusionEmbeddingDataset.to_device(): tensors pre-loaded to GPU')
print('  VRAM peak: 1.001 GB @ batch=4096 (model ~563K params)')
print('=' * 65)

## 5. Full AUC Progression (Phase 3 → 11)

In [ ]:
auc_history = [
    ('Phase 3\nPrice CNN',                    0.5504),
    ('Phase 4\nGAT',                          0.5777),
    ('Phase 5\nMacro MLP',                    0.5031),
    ('Phase 6\nV2 Fusion',                    0.5989),
    ('Phase 11B\nV2.1 (128d)',                fusion_v2_1['direction_auc']),
    (f'Phase 11A\nV2.1+Filter\n({v2_1_best_cov*100:.0f}% cov)', v2_1_best_f_auc),
]

labels = [x[0] for x in auc_history]
aucs   = [x[1] for x in auc_history]
colors = ['#4878CF', '#4878CF', '#6ACC65', '#B47CC7', '#FF7F0E', '#FF3333']

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(labels, aucs, color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)
ax.axhline(0.50, color='red',   linestyle='--', linewidth=1.2, label='Random (AUC=0.50)')
ax.axhline(0.60, color='green', linestyle=':',  linewidth=1.2, label='Target (AUC=0.60)')
for bar, auc in zip(bars, aucs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
            f'{auc:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylabel('ROC-AUC')
ax.set_ylim(0.40, 0.72)
ax.set_title('AUC Progression: Phase 3 → 11')
ax.legend(loc='upper left')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../models/phase11_auc_progression.png', bbox_inches='tight')
plt.show()
print('Figure saved to models/phase11_auc_progression.png')